In [7]:
import numpy as np
import os
import sys
import time
import random
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
import numpy as np
import sys
import os
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os

PROJECT_ROOT = "/home/jupyter/project/Student-Camp-Projects"
sys.path.append(PROJECT_ROOT)

from src.dataset_utils.dataset import ACTClusterDataset

In [8]:
def set_seed(seed: int = 42):
    random.seed(seed)                       
    np.random.seed(seed)                   
    torch.manual_seed(seed)             
    torch.cuda.manual_seed(seed)         
    torch.cuda.manual_seed_all(seed)        

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 128

In [11]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 1)
        )
        
    def forward(self, images, redshift):
        return self.cnn(images)

In [12]:
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
loss_fn = torch.nn.MSELoss()

In [13]:
train_indices = np.load("/home/jupyter/project/Student-Camp-Projects/src/data/augm_norm/train_indices.npy")
test_indices = np.load("/home/jupyter/project/Student-Camp-Projects/src/data/augm_norm/test_indices.npy")

In [14]:
train_dataset = ACTClusterDataset("/home/jupyter/project/Student-Camp-Projects/dataset/", train_indices, norm=True, augm=True)
test_dataset = ACTClusterDataset("/home/jupyter/project/Student-Camp-Projects/dataset/", test_indices, norm=True, augm=True)

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,     
    pin_memory=True,   
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,     
    pin_memory=True,    
)

In [16]:
model.train()
losses = []

for epoch in range(EPOCHS):
    start_t = time.time()
    total_loss = 0
    for img, scalars, target in train_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        pred = model(img, scalars)
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss / len(train_loader)) 

    scheduler.step()

    if epoch == 0:
        print("Epoch 1 time:", time.time() - start_t)

    # --- оценка каждые 10 эпох ---
    if (epoch + 1) % 10 == 0:
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for img, scalars, target in test_loader:
                img = img.to(device)
                scalars = scalars.to(device)
                target = target.to(device)

                pred = model(img, scalars)
                preds.append(pred)
                targets.append(target)

        model.train()

        preds = torch.cat(preds)
        targets = torch.cat(targets)

        mae = torch.abs(targets - preds)
        mean_mae = mae.mean().item()
        max_mae = mae.max().item()

        relative_errors = mae / targets
        mean_rel_error = relative_errors.mean().item()
        max_rel_error = relative_errors.max().item()

        print(f"EPOCH={epoch+1}")
        print(f"Средняя MAE {mean_mae:.4f}")
        print(f"Максимальная MAE {max_mae:.4f}")
        print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
        print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")
losses = np.asarray(losses)

EPOCH=80
Средняя MAE 0.1600
Максимальная MAE 0.8378
Средняя относительная ошибка: 0.2871
Максимальная относительная ошибка: 1.6691


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(

EPOCH=90
Средняя MAE 0.1601
Максимальная MAE 0.8368
Средняя относительная ошибка: 0.2877
Максимальная относительная ошибка: 1.6743


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(

EPOCH=100
Средняя MAE 0.1601
Максимальная MAE 0.8373
Средняя относительная ошибка: 0.2874
Максимальная относительная ошибка: 1.6726


In [17]:
model.eval()  
preds = []
targets = []

with torch.no_grad():
    for img, scalars, target in test_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        pred = model(img, scalars)

        preds.append(pred.cpu())
        targets.append(target.cpu())

preds = torch.cat(preds)
targets = torch.cat(targets)

mae = torch.abs(targets - preds)
mean_mae = mae.mean().item()
max_mae = mae.max().item()

relative_errors = mae / torch.abs(targets)
mean_rel_error = relative_errors.mean().item()
max_rel_error = relative_errors.max().item()

print(f"EPOCH={epoch+1}")
print(f"Средняя MAE: {mean_mae:.4f}")
print(f"Максимальная MAE: {max_mae:.4f}")
print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")

EPOCH=100
Средняя MAE: 0.1601
Максимальная MAE: 0.8373
Средняя относительная ошибка: 0.2874
Максимальная относительная ошибка: 1.6726


In [18]:
output_dir = "/home/jupyter/project/Student-Camp-Projects/src/results/cnn/norm_augm/"
np.save(output_dir + "losses.npy", losses)
np.save(output_dir + "targets.npy", targets)
np.save(output_dir + "preds.npy", preds)

In [19]:
output_dir = "/home/jupyter/project/Student-Camp-Projects/src/results/cnn/norm_augm/"
np.save(output_dir + "norm_augm_losses.npy", losses)
np.save(output_dir + "norm_augm_targets.npy", targets)
np.save(output_dir + "norm_augm_preds.npy", preds)